# Multi-Class LLM Evaluation — UNSW-NB15

Mirrors `1-cic-iot/03-multiclass-classification-llm.ipynb`.

**Dataset:** UNSW-NB15 (10 named attack categories).
Worms excluded (174 samples — too few for reliable per-class F1).
Remaining 9 classes balanced to `TARGET_PER_CLASS` samples via subsampling.

**Method:** Per-class LangChain feedback loop with `claude-haiku-4-5-20251001`.
Rules are inequality-based thresholds; `==` and `!=` are forbidden.

**Output:** `results/ml/` (DT, RF baselines) + `results/llm/` (LLM rules + report).

In [1]:
################################################################################
# Cell 1 — Load Dataset and Build Per-Class Splits
#
# Sources: two raw UNSW-NB15 CSV files (training + testing) concatenated.
# Label column: 'attack_cat' (10 named categories).
# Numeric features only; drop ['label', 'attack_cat', 'id'].
# Exclude Worms (174 samples). Balance all remaining classes to TARGET_PER_CLASS.
# Stratified 80/20 split per class.
################################################################################

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tabulate import tabulate

dataset_name = 'unsw-nb15'
TRAIN_PATH   = os.path.expanduser(
    '~/Documents/Projects/RAG Paper/data/unsw-nb15/UNSW_NB15_training-set.csv'
)
TEST_PATH    = os.path.expanduser(
    '~/Documents/Projects/RAG Paper/data/unsw-nb15/UNSW_NB15_testing-set.csv'
)
LABEL_COL    = 'attack_cat'
DROP_COLS    = ['label', 'attack_cat', 'id']
EXCLUDE_CLASSES = ['Worms']   # too few samples for reliable evaluation

df_train_raw = pd.read_csv(TRAIN_PATH, encoding='utf-8-sig')
df_test_raw  = pd.read_csv(TEST_PATH,  encoding='utf-8-sig')
df_raw       = pd.concat([df_train_raw, df_test_raw], ignore_index=True)
df_raw[LABEL_COL] = df_raw[LABEL_COL].str.strip()

feature_cols = [
    c for c in df_raw.select_dtypes(include=[np.number]).columns
    if c not in DROP_COLS
]
df = df_raw[feature_cols + [LABEL_COL]].copy()
df = df[~df[LABEL_COL].isin(EXCLUDE_CLASSES)].reset_index(drop=True)

print(f'Population after exclusions: {len(df):,} rows, {len(feature_cols)} features')
print(f'Classes: {sorted(df[LABEL_COL].unique())}')
print()
label_counts = df[LABEL_COL].value_counts()
print('Class distribution:')
print(label_counts.to_string())

# Balance: subsample each class to min class count
TARGET_PER_CLASS = int(label_counts.min())   # = 1511 (Shellcode)
print(f'\nTARGET_PER_CLASS = {TARGET_PER_CLASS} (min class count)')

train_dfs, test_dfs = {}, {}
for label in sorted(df[LABEL_COL].unique()):
    cls_df = df[df[LABEL_COL] == label].sample(
        n=min(TARGET_PER_CLASS, len(df[df[LABEL_COL] == label])),
        random_state=42
    )
    train_part = cls_df.sample(frac=0.8, random_state=42)
    test_part  = cls_df.drop(train_part.index)
    train_dfs[label] = train_part.drop(columns=[LABEL_COL]).reset_index(drop=True)
    test_dfs[label]  = test_part.drop(columns=[LABEL_COL]).reset_index(drop=True)

X_train_all = pd.concat(
    [df.assign(label=lbl) for lbl, df in train_dfs.items()]
).reset_index(drop=True)
X_test_all  = pd.concat(
    [df.assign(label=lbl) for lbl, df in test_dfs.items()]
).reset_index(drop=True)

table_data = [
    [lbl, len(train_dfs[lbl]) + len(test_dfs[lbl]),
     len(train_dfs[lbl]), len(test_dfs[lbl])]
    for lbl in sorted(train_dfs)
]
print()
print(tabulate(table_data, headers=['Class', 'Total', 'Train', 'Test'], tablefmt='grid'))
print(f'\nFeatures ({len(feature_cols)}): {feature_cols}')


Population after exclusions: 257,499 rows, 39 features
Classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode']

Class distribution:
attack_cat
Normal            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511

TARGET_PER_CLASS = 1511 (min class count)

+----------------+---------+---------+--------+
| Class          |   Total |   Train |   Test |
+================+=========+=========+========+
| Analysis       |    1511 |    1209 |    302 |
+----------------+---------+---------+--------+
| Backdoor       |    1511 |    1209 |    302 |
+----------------+---------+---------+--------+
| DoS            |    1511 |    1209 |    302 |
+----------------+---------+---------+--------+
| Exploits       |    1511 |    1209 |    302 |
+----------------+---------+---------+--------+
| Fuzzers

## ML Baseline (Multi-Class)

In [2]:
################################################################################
# Cell 2 — Decision Tree and Random Forest Multi-Class Baselines
#
# class_weight='balanced' to handle residual imbalance.
# Categorical columns label-encoded before fitting.
################################################################################

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import time, os

os.makedirs('results/ml', exist_ok=True)

# Encode categorical features (UNSW-NB15 numeric-only, but guard just in case)
le_map = {}
X_tr = X_train_all[feature_cols].copy()
X_te = X_test_all[feature_cols].copy()
for col in X_tr.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X_tr[col] = le.fit_transform(X_tr[col].astype(str))
    X_te[col] = X_te[col].astype(str).map(
        dict(zip(le.classes_, le.transform(le.classes_)))
    ).fillna(-1).astype(int)
    le_map[col] = le

y_train_lbl = X_train_all['label'].values
y_test_lbl  = X_test_all['label'].values

for ModelClass, fname in [
    (DecisionTreeClassifier, 'result-dt-multiclass.txt'),
    (RandomForestClassifier,  'result-rf-multiclass.txt'),
]:
    kwargs = {'random_state': 42, 'class_weight': 'balanced'}
    if ModelClass == RandomForestClassifier:
        kwargs['n_estimators'] = 100
    model = ModelClass(**kwargs)
    model.fit(X_tr.values, y_train_lbl)
    t0 = time.time()
    y_pred = model.predict(X_te.values)
    elapsed = time.time() - t0
    report = classification_report(y_test_lbl, y_pred, digits=4)
    matrix = confusion_matrix(y_test_lbl, y_pred)
    print(f'=== {ModelClass.__name__} ({elapsed:.4f}s) ===')
    print(report)
    with open(f'results/ml/{fname}', 'w') as f:
        f.write(f'Classification Report\n{report}\n\nConfusion Matrix\n{matrix}\n')


=== DecisionTreeClassifier (0.0006s) ===
                precision    recall  f1-score   support

      Analysis     0.2635    0.3543    0.3023       302
      Backdoor     0.2107    0.2086    0.2097       302
           DoS     0.2247    0.2351    0.2298       302
      Exploits     0.5399    0.4702    0.5027       302
       Fuzzers     0.7196    0.6457    0.6806       302
       Generic     0.9835    0.9868    0.9851       302
        Normal     0.8092    0.8146    0.8119       302
Reconnaissance     0.8303    0.7616    0.7945       302
     Shellcode     0.8853    0.8179    0.8503       302

      accuracy                         0.5883      2718
     macro avg     0.6074    0.5883    0.5963      2718
  weighted avg     0.6074    0.5883    0.5963      2718

=== RandomForestClassifier (0.0214s) ===
                precision    recall  f1-score   support

      Analysis     0.3135    0.3311    0.3221       302
      Backdoor     0.2625    0.2252    0.2424       302
           DoS    

## Vector Store — Per-Class Representative Samples

In [3]:
################################################################################
# Cell 3 — Per-Class Representative Samples via BGE-M3
#
# For each class: subsample MAX_EMBED rows, embed with BAAI/bge-m3,
# compute mean vector, retrieve top N_REPR rows by cosine similarity to mean.
# Stores as class_entries[label] = {feature_name: [list of 10 values]}.
################################################################################

import json
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from tqdm import tqdm

N_REPR    = 10
MAX_EMBED = 100

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'mps'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 64},
)


def get_representative_samples_bge(
    df: pd.DataFrame, cols: list, n: int = 10,
    max_embed: int = 100, seed: int = 42
) -> pd.DataFrame:
    """Embed up to max_embed rows, return n closest to the class mean."""
    sample = df[cols].sample(n=min(max_embed, len(df)), random_state=seed)
    docs   = [str(row.tolist()) for _, row in sample.iterrows()]
    vecs   = np.array(embeddings.embed_documents(docs))
    mean_v = vecs.mean(axis=0)
    norms  = np.linalg.norm(vecs, axis=1) * np.linalg.norm(mean_v)
    sims   = (vecs @ mean_v) / np.where(norms == 0, 1e-9, norms)
    top_i  = np.argsort(sims)[::-1][:n]
    return sample.iloc[top_i]


class_entries = {}
for label in tqdm(sorted(train_dfs), desc='Building class entries'):
    cls_df  = train_dfs[label]
    top_rows = get_representative_samples_bge(cls_df, feature_cols, N_REPR, MAX_EMBED)
    class_entries[label] = {col: top_rows[col].tolist() for col in feature_cols}

print('class_entries keys:', list(class_entries.keys()))


Building class entries: 100%|██████████| 9/9 [00:39<00:00,  4.40s/it]

class_entries keys: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode']


## Tool Definition

In [4]:
################################################################################
# Cell 4 — evaluate_rule Tool (multi-class, one-vs-rest)
#
# Returns macro F1 for target_class vs all other classes on training data.
# Operators restricted to >, <, >=, <= (no == or !=).
################################################################################

from sklearn.metrics import classification_report
from tqdm import tqdm
import operator as op_module
from typing import Annotated
from langchain_core.tools import tool

show_progress = False
operations = {
    '<': op_module.lt, '>': op_module.gt,
    '<=': op_module.le, '>=': op_module.ge,
}


@tool
def evaluate_rule(
    feature_name: Annotated[str, 'Feature name'],
    value:        Annotated[str, 'Threshold value'],
    op:           Annotated[str, 'Operator (>, <, >=, <=)'],
    target_class: Annotated[str, 'Target class name'],
) -> float:
    """Evaluate rule on training set. Returns macro F1 for target_class vs all others."""
    try:
        value = float(value)
    except (ValueError, TypeError):
        pass
    if op not in operations:
        raise ValueError(f'Unsupported operator: {op}  (use >, <, >=, <= only)')
    y_true, y_pred = [], []
    for lbl, df_cls in train_dfs.items():
        for i in tqdm(range(len(df_cls)), disable=not show_progress,
                      desc=f'Eval {lbl[:12]}...'):
            y_true.append('target' if lbl == target_class else 'other')
            try:
                y_pred.append(
                    'target' if operations[op](df_cls.iloc[i][feature_name], value)
                    else 'other'
                )
            except (KeyError, TypeError):
                y_pred.append('other')
    report = classification_report(
        y_true, y_pred, digits=4, output_dict=True, zero_division=0
    )
    return report['macro avg']['f1-score']


## Prompt Template

In [5]:
################################################################################
# Cell 5 — Prompt Template
################################################################################

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_message = ('system',
"""You are a network security analyst specialising in IoT intrusion detection.
You are given labelled network traffic data with named features and numeric values.
Your task is to generate exactly {k} deterministic threshold rules to identify '{target_class}' traffic.

Rules must:
- Use ONLY inequality operators: '>', '<', '>=', '<='
- NEVER use '==' or '!=' \u2014 these are forbidden for numeric features
- Reference only feature names present in the data
- Be range-based thresholds that generalise beyond the exact sample values shown
- You MUST make exactly {k} tool calls \u2014 one per rule, no more, no less.

Good rule example: dur < 0.01
Bad rule example: proto == 6  \u2190 FORBIDDEN""")

human_message = ('user',
"""Analyze the following network data and generate rules for the top {k} important features \
to identify '{target_class}' entries.

Target Class Entries:
```{target_entries}```

Other Class Entries (sample):
```{other_entries}```""")

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    MessagesPlaceholder('msgs'),
])


## LLM Setup

In [6]:
################################################################################
# Cell 6 — LLM Setup and Tool Binding
################################################################################

import os, dotenv
from langchain_anthropic import ChatAnthropic

dotenv.load_dotenv(os.getcwd() + '/../.env')

model_name = 'claude-haiku-4-5-20251001'
llm = ChatAnthropic(model=model_name, temperature=0.1)
llm_with_tool = llm.bind_tools([evaluate_rule])

chain = prompt | llm_with_tool
print(f'LLM: {model_name}')


LLM: claude-haiku-4-5-20251001


## Feedback Loop (Per Class)

In [10]:
################################################################################
# Cell 13 — Per-Class Feedback Loop (with ValidationError handling + resume)
#
# - Resumes from existing class_rules-multiclass.json if present (skips done classes)
# - Wraps evaluate_rule.invoke() in try/except to handle missing-field ValidationErrors
# - Saves checkpoint after each class so a mid-run crash loses at most one class
################################################################################

import json, os
from langchain_core.messages import HumanMessage

try:
    from pydantic import ValidationError as PydanticValidationError
except ImportError:
    PydanticValidationError = ValueError

CLASS_CONFIG = {
    'Generic':        {'max_rounds': 15, 'k': 5},
    'Exploits':       {'max_rounds': 15, 'k': 5},
    'Fuzzers':        {'max_rounds': 12, 'k': 5},
    'DoS':            {'max_rounds': 12, 'k': 5},
    'Reconnaissance': {'max_rounds': 12, 'k': 5},
    'Analysis':       {'max_rounds': 15, 'k': 7},
    'Backdoor':       {'max_rounds': 15, 'k': 7},
    'Shellcode':      {'max_rounds': 15, 'k': 7},
}

patience       = 4
show_progress  = False

os.makedirs('results/llm', exist_ok=True)
rules_output_path = 'results/llm/class_rules-multiclass.json'

# Resume: load existing results and skip completed classes
if os.path.exists(rules_output_path):
    with open(rules_output_path) as f:
        class_rules = json.load(f)
    print(f'Resuming — already done: {list(class_rules.keys())}')
else:
    class_rules = {}

non_normal_classes = [lbl for lbl in sorted(train_dfs) if lbl != 'Normal']


def build_other_entries(target_class, class_entries, rows_per_class=3):
    other = {}
    for cls, entries in class_entries.items():
        if cls != target_class:
            other[cls] = {k: v[:rows_per_class] for k, v in entries.items()}
    return other


def safe_invoke_rule(tool_call):
    """Invoke evaluate_rule; return score=0.0 on ValidationError (missing field)."""
    try:
        tool_msg = evaluate_rule.invoke(tool_call)
        return tool_msg, float(tool_msg.content)
    except (PydanticValidationError, Exception) as exc:
        if 'Field required' in str(exc) or 'ValidationError' in type(exc).__name__:
            from langchain_core.messages import ToolMessage
            dummy = ToolMessage(content='0.0', tool_call_id=tool_call.get('id', 'unknown'))
            print(f'    [WARN] Malformed tool call (missing field) — scoring 0.0: {exc}')
            return dummy, 0.0
        raise


for target_class in non_normal_classes:
    if target_class in class_rules:
        print(f'Skipping {target_class} (already in checkpoint)')
        continue

    print(f'\n=== Target class: {target_class} ===')

    cfg        = CLASS_CONFIG.get(target_class, {'max_rounds': 12, 'k': 5})
    max_rounds = cfg['max_rounds']
    k          = cfg['k']

    n = no_improve = 0
    best_mean_f1    = 0.0
    best_tool_calls = []
    train_f1_scores = []
    msgs            = []

    target_entries = json.dumps(class_entries[target_class])
    other_entries  = json.dumps(build_other_entries(target_class, class_entries))

    while n < max_rounds and no_improve < patience:
        ai_msg = chain.invoke({
            'k': k, 'target_class': target_class,
            'target_entries': target_entries,
            'other_entries': other_entries,
            'msgs': msgs,
        })

        tool_msgs   = []
        rule_scores = []
        for tool_call in ai_msg.tool_calls:
            tool_msg, score = safe_invoke_rule(tool_call)
            tool_msgs.append(tool_msg)
            rule_scores.append(score)

        mean_f1 = sum(rule_scores) / len(rule_scores) if rule_scores else 0.0

        if mean_f1 > best_mean_f1:
            best_mean_f1    = mean_f1
            best_tool_calls = list(ai_msg.tool_calls)
            no_improve      = 0
        else:
            no_improve += 1

        rule_feedback_lines = []
        for i, (tc, score) in enumerate(zip(ai_msg.tool_calls, rule_scores)):
            args = tc['args']
            rule_feedback_lines.append(
                f"  Rule {i+1}: {args.get('feature_name','?')} {args.get('op','?')} {args.get('value','?')} "
                f"-> F1={score:.4f} ({'KEEP' if score >= mean_f1 else 'REVISE'})"
            )
        rule_feedback = '\n'.join(rule_feedback_lines)

        human_msg = HumanMessage(
            f"Round {n+1} results (best so far: {best_mean_f1:.4f}):\n"
            f"{rule_feedback}\n\n"
            f"Mean F1 this round: {mean_f1:.4f}. "
            f"Revise rules marked REVISE by trying different threshold values or a different feature. "
            f"Keep rules marked KEEP unchanged. "
            f"Generate exactly {k} rules for '{target_class}' and make exactly {k} tool calls."
        )

        msgs.extend([ai_msg, *tool_msgs, human_msg])
        train_f1_scores.append(mean_f1)
        n += 1

        usage_info = ai_msg.response_metadata.get('usage', {})
        print(f'  Round: {n}  Mean F1: {mean_f1:.4f}  Best: {best_mean_f1:.4f}  '
              f'No-improve: {no_improve}  '
              f'Tokens: {usage_info.get("input_tokens", 0) + usage_info.get("output_tokens", 0)}')

    class_rules[target_class] = [{'args': tc['args']} for tc in best_tool_calls]
    print(f'  Train F1 history: {train_f1_scores}')
    print(f'  Final best F1: {best_mean_f1:.4f}  (stopped after {n} rounds)')

    # Checkpoint: save after each class so a crash loses at most one class
    with open(rules_output_path, 'w') as f:
        json.dump(class_rules, f, indent=2)
    print(f'  [Checkpoint saved — {len(class_rules)} classes]')


print('\nDone. Classes with rules:', list(class_rules.keys()))
print(f'Saved class rules to: {rules_output_path}')


Resuming — already done: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode']
Skipping Analysis (already in checkpoint)
Skipping Backdoor (already in checkpoint)
Skipping DoS (already in checkpoint)
Skipping Exploits (already in checkpoint)
Skipping Fuzzers (already in checkpoint)
Skipping Generic (already in checkpoint)
Skipping Reconnaissance (already in checkpoint)
Skipping Shellcode (already in checkpoint)

Done. Classes with rules: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode']
Saved class rules to: results/llm/class_rules-multiclass.json


In [8]:
################################################################################
# Recovery Cell — Save partial class_rules to JSON
#
# Run this immediately if cell 13 failed mid-loop.
# Saves whatever is in `class_rules` plus tries to capture Shellcode's
# best_tool_calls that are still in kernel memory.
################################################################################

import json, os

# Try to capture Shellcode's partial best state from the failed run
try:
    if 'best_tool_calls' in dir() and best_tool_calls and target_class == 'Shellcode':
        class_rules['Shellcode'] = [{'args': tc['args']} for tc in best_tool_calls]
        print(f'  Captured Shellcode rules from in-memory best_tool_calls (F1={best_mean_f1:.4f})')
except Exception as e:
    print(f'  Could not capture Shellcode state: {e}')

os.makedirs('results/llm', exist_ok=True)
out = 'results/llm/class_rules-multiclass.json'
with open(out, 'w') as f:
    json.dump(class_rules, f, indent=2)

print(f'Saved {len(class_rules)} classes to {out}')
print('Classes saved:', list(class_rules.keys()))


  Captured Shellcode rules from in-memory best_tool_calls (F1=0.4986)
Saved 8 classes to results/llm/class_rules-multiclass.json
Classes saved: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode']


## Evaluate All Rules on Test Set (Multi-Class)

In [11]:
################################################################################
# Cell 8 — Multi-Class Test Set Evaluation
#
# For each test row: compute rule-match score per class (proportion of rules
# that fired). Predict class with highest score; tie-break to 'Normal'.
################################################################################

import json, operator
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

operations_eval = {
    '<': operator.lt, '>': operator.gt,
    '<=': operator.le, '>=': operator.ge,
    '==': operator.eq, '!=': operator.ne,
}

os.makedirs('results/llm', exist_ok=True)
rules_input_path = 'results/llm/class_rules-multiclass.json'
with open(rules_input_path) as f:
    class_rules = json.load(f)

print(f'Loaded {len(class_rules)} classes, rule counts: '
      f'{ {c: len(r) for c, r in class_rules.items()} }')


def predict_multiclass(row, class_rules):
    scores = {}
    for cls, tool_calls in class_rules.items():
        if not tool_calls:
            scores[cls] = 0.0
            continue
        count = 0
        for tc in tool_calls:
            args = tc['args']
            op, feat, val = args['op'], args['feature_name'], args['value']
            try:
                val = float(val)
            except (ValueError, TypeError):
                pass
            if op in operations_eval and feat in row.index:
                try:
                    if operations_eval[op](row[feat], val):
                        count += 1
                except TypeError:
                    pass
        scores[cls] = count / len(tool_calls)

    max_score = max(scores.values()) if scores else 0.0
    if max_score == 0.0:
        return 'Normal'
    winners = [cls for cls, s in scores.items() if s == max_score]
    return winners[0] if len(winners) == 1 else 'Normal'


# Build flat test set (all classes including Normal)
test_rows_all, test_labels_all = [], []
for label, df_cls in test_dfs.items():
    for i in range(len(df_cls)):
        test_rows_all.append(df_cls.iloc[i])
        test_labels_all.append(label)

y_pred_llm, y_true_llm = [], []
for i in tqdm(range(len(test_rows_all)), desc='Evaluating multiclass rules'):
    y_true_llm.append(test_labels_all[i])
    y_pred_llm.append(predict_multiclass(test_rows_all[i], class_rules))

report = classification_report(y_true_llm, y_pred_llm, digits=4)
matrix = confusion_matrix(y_true_llm, y_pred_llm)
print(report)
with open('results/llm/result-llm-multiclass.txt', 'w') as f:
    f.write(f'Classification Report\n{report}\n\nConfusion Matrix\n{matrix}\n')


Loaded 8 classes, rule counts: {'Analysis': 7, 'Backdoor': 7, 'DoS': 5, 'Exploits': 5, 'Fuzzers': 5, 'Generic': 5, 'Reconnaissance': 5, 'Shellcode': 7}


Evaluating multiclass rules: 100%|██████████| 2718/2718 [00:00<00:00, 18831.72it/s]

                precision    recall  f1-score   support

      Analysis     0.0780    0.0364    0.0497       302
      Backdoor     0.1330    0.0894    0.1069       302
           DoS     0.0727    0.0265    0.0388       302
      Exploits     0.1103    0.0497    0.0685       302
       Fuzzers     0.5000    0.0232    0.0443       302
       Generic     1.0000    0.0199    0.0390       302
        Normal     0.0495    0.3013    0.0850       302
Reconnaissance     1.0000    0.0066    0.0132       302
     Shellcode     0.2180    0.1921    0.2042       302

      accuracy                         0.0828      2718
     macro avg     0.3513    0.0828    0.0722      2718
  weighted avg     0.3513    0.0828    0.0722      2718



## Efficiency Comparison

In [12]:
################################################################################
# Cell 9 — Inference Time Comparison
################################################################################

import time
from tabulate import tabulate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Refit models on encoded train set (already done above, reusing X_tr, y_train_lbl)
model_dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')
model_rf = RandomForestClassifier(n_estimators=100, random_state=42,
                                   class_weight='balanced', n_jobs=-1)
model_dt.fit(X_tr.values, y_train_lbl)
model_rf.fit(X_tr.values, y_train_lbl)

n_samples = min(1000, len(X_te))
elapsed_dt, elapsed_rf, elapsed_llm = [], [], []

for i in range(n_samples):
    row_arr = X_te.iloc[[i]].values

    t0 = time.time()
    model_dt.predict(row_arr)
    elapsed_dt.append(time.time() - t0)

    t0 = time.time()
    model_rf.predict(row_arr)
    elapsed_rf.append(time.time() - t0)

    t0 = time.time()
    predict_multiclass(X_te.iloc[i], class_rules)
    elapsed_llm.append(time.time() - t0)

rows = [
    ['Decision Tree', f'{sum(elapsed_dt)/n_samples*1e3:.4f} ms'],
    ['Random Forest', f'{sum(elapsed_rf)/n_samples*1e3:.4f} ms'],
    ['LLM Rules',     f'{sum(elapsed_llm)/n_samples*1e3:.4f} ms'],
]
print(tabulate(rows, headers=['Model', 'Avg Inference Time'], tablefmt='grid'))


+---------------+----------------------+
| Model         | Avg Inference Time   |
+===============+======================+
| Decision Tree | 0.1034 ms            |
+---------------+----------------------+
| Random Forest | 13.6929 ms           |
+---------------+----------------------+
| LLM Rules     | 0.1260 ms            |
+---------------+----------------------+
